# 15. 🚀 Продвинутое: графы, спектральная теория и GNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/15_graphs_gnn.ipynb)

Код из раздела [modules/15_graphs_gnn.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/15_graphs_gnn.md#m15). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Graph Neural Networks

In [ ]:
import numpy as np
def gcn_layer(A, H, W):
    A_hat = A + np.eye(len(A))                                   # петли: вершина видит себя
    d = A_hat.sum(1); D_inv_sqrt = np.diag(d ** -0.5)
    return np.maximum(0, D_inv_sqrt @ A_hat @ D_inv_sqrt @ H @ W)

A = np.array([[0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]], float)
H = np.eye(4)                                                    # признаки = one-hot вершины
print(gcn_layer(A, H, np.random.randn(4, 2)).round(2))

### 🏋️ Практика модуля

**15.1.** Для графа из начала модуля найдите собственные числа $L$ и проверьте: сколько нулей и что говорит вектор Фидлера.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
A = np.array([[0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]], float)
L = np.diag(A.sum(1)) - A
w, V = np.linalg.eigh(L)
print(w.round(3))          # [0. 1. 3. 4.] — один ноль: граф связный
print(V[:, 1].round(2))    # знаки разделяют {1, 2} и {4}, вершина 3 — «мост» с промежуточным значением

### 🏋️ Практика модуля

**15.2.** Посчитайте PageRank для того же графа степенным методом.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
P = A / A.sum(1, keepdims=True)
r = np.full(4, 0.25)
for _ in range(100):
    r = 0.85 * P.T @ r + 0.15 / 4
print(r.round(3))          # у вершины 3 (степень 3) наибольший ранг

### 🏋️ Практика модуля

**15.3.** Покажите численно over-smoothing: примените 50 раз нормированную смежность (без весов и активаций) к случайным признакам.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
A_hat = A + np.eye(4); d = A_hat.sum(1)
S = np.diag(d ** -0.5) @ A_hat @ np.diag(d ** -0.5)
H = np.random.randn(4, 3)
for k in [1, 5, 50]:
    Hk = np.linalg.matrix_power(S, k) @ H
    Hn = Hk / np.linalg.norm(Hk, axis=1, keepdims=True)
    print(k, np.round(Hn @ Hn.T, 2).min())   # косинусная близость между вершинами → 1

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def spectral_clustering():
    rng = np.random.default_rng(0)
    n = 150
    t = rng.uniform(0, np.pi, n)
    A = np.c_[np.cos(t), np.sin(t)] + 0.06 * rng.normal(size=(n, 2))
    B = np.c_[1 - np.cos(t), 0.5 - np.sin(t)] + 0.06 * rng.normal(size=(n, 2))
    X = np.r_[A, B]
    D2 = ((X[:, None] - X[None]) ** 2).sum(-1)
    Wm = np.exp(-D2 / (2 * 0.1**2)); np.fill_diagonal(Wm, 0)
    L = np.diag(Wm.sum(1)) - Wm
    Dm = np.diag(1 / np.sqrt(Wm.sum(1)))
    w, V = np.linalg.eigh(Dm @ L @ Dm)
    fied = V[:, 1]
    # k-means на исходных координатах для сравнения
    c = X[[0, n]]
    for _ in range(20):
        lab = ((X[:, None] - c[None]) ** 2).sum(-1).argmin(1)
        c = np.array([X[lab == j].mean(0) for j in range(2)])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(*X.T, c=lab, cmap="coolwarm", s=10); axes[0].set_title("k-means: режет «по прямой»")
    axes[1].scatter(*X.T, c=fied > 0, cmap="coolwarm", s=10); axes[1].set_title("Спектральная: знак вектора Фидлера")
    axes[2].plot(np.sort(fied), color=C[0], lw=2); axes[2].axhline(0, color="k", lw=0.6)
    axes[2].set_title(f"Отсортированный 2-й собственный вектор лапласиана\nλ₁={w[0]:.3f}, λ₂={w[1]:.3f}, λ₃={w[2]:.3f}")
    for ax in axes[:2]: ax.set_aspect("equal")
    save(fig, "spectral_clustering")

spectral_clustering()